## Stage 03b — MML Instrument Alignment

Aligns MML platform instruments using two corrections applied together:

1. **Tube delay** (Sections E1–E3): gas analyzers sample through a tube inlet.
   The H2O spike (e.g., exhaled breath near sensors) arrives at the Anem instantly
   but is delayed at the gas inlet by the tube travel time.
   Reference = Anem wind components (u/v/w) + RH_pct (second breath-spike proxy)
   + pressure_mbar (context only); test = gas H2O_ppm (+ other species).
   Correction shifts gas timestamps *earlier* to remove the tube delay.

2. **GPS clock correction** (Section F): the LANL Toughbook clock drifted over the
   campaign (fast by ~0.7 s in January, ~13 s by March 8).
   GPS satellite UTC is the ground truth.  `gps_corrections[date_tag]` = median
   `(toughbook_epoch − GPS_UTC_s)` per date; positive = toughbook fast.

**Total lag applied to gas:** `tube_lag − gps_corr`
**Total lag applied to Anem / GPS:** `−gps_corr`

> Run `03_survey.ipynb` first. Files on dates you marked `bad` are pre-rejected
> and auto-skipped in the widget.  Files on `uncertain` dates show `[?]` in the title.

| Section | Instrument | Primary ref | Secondary gas ref | Method |
|---|---|---|---|---|
| E1 | LANL_aerisultra321 | Anem u/v/w/RH/P | — | manual H2O spike |
| E2 | LANL_aerispico017  | Anem u/v/w/RH/P | Ultra321 H2O aligned | manual H2O spike |
| E3 | UOU_LGR (Mar 10)   | Anem u/v/w/RH/P | Ultra321 H2O aligned | manual H2O spike |
| F  | LANL_GPS | GPS satellite UTC | — | auto median offset |

**Secondary gas ref** (Sections E2/E3): Ultra 321 H2O_ppm from `03_instrument_aligned/`
(already gps+tube corrected). Shows as a dashed blue trace — helps confirm the spike
location when Ultra 321 and Pico/LGR both captured the same H2O event.

**Outputs:** `lag_offsets_mml.json`, `apply_manifest_mml.json`,
aligned Parquet in `03_instrument_aligned/`.

In [ ]:
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

sys.path.insert(0, str(Path().resolve().parent))
from paths import STAGE_02_DIR, STAGE_03_DIR, QUALITY_MANIFEST_PATH, REPO_ROOT
from src.provenance import git_info, check_clean, upstream_ref

display(HTML('<style>.plotly-graph-div { width: 100% !important; }</style>'))
print('Imports OK')

In [ ]:
MAX_LAG_S      = 1800
MAX_GPS_CORR_S = 60   # files with |offset| > 60s treated as no GPS lock

ULTRA321_DIR = STAGE_02_DIR / 'LANL_aerisultra321' / 'Raw'
PICO017_DIR  = STAGE_02_DIR / 'LANL_aerispico017'  / 'Raw'
LGR_DIR      = STAGE_02_DIR / 'UOU_LGR'
ANEM_DIR     = STAGE_02_DIR / 'LANL_Anem'
GPS_DIR      = STAGE_02_DIR / 'LANL_GPS'

with open(STAGE_02_DIR / 'routing_manifest.json') as _fh:
    ROUTING = json.load(_fh)
_mml = sum(1 for v in ROUTING.values() if v == 'MML')
_wyo = sum(1 for v in ROUTING.values() if v == 'WYO')
print(f'Routing manifest: {len(ROUTING)} entries  (MML={_mml}, WYO={_wyo})')
print('Config OK')

In [ ]:
from src.align import (
    resample_series, cross_correlate, raw_stem, date_tag,
    apply_lag_to_parquet, load_quality_manifest, file_quality,
    load_aligned_series, reference_bad_dates,
)


def load_parquet_col(path, col):
    return pd.read_parquet(path, columns=[col])[col].dropna()


def save_lag_offsets_mml():
    STAGE_03_DIR.mkdir(parents=True, exist_ok=True)
    g = globals()
    def _lags(conf, rej):
        return {k: v for k, v in conf.items() if k not in rej}
    git_hash, git_dirty = git_info(REPO_ROOT)
    manifest = {
        'stage':    '03b_align_mml',
        'run_utc':  datetime.now(timezone.utc).isoformat(),
        'git_hash': git_hash,
        'git_dirty': git_dirty,
        'upstream': upstream_ref(STAGE_02_DIR / 'run_manifest.json'),
        'tube_lags': {
            'LANL_aerisultra321': _lags(g.get('u321_mml_confirmed', {}), g.get('u321_mml_rejected', set())),
            'LANL_aerispico017':  _lags(g.get('pico_mml_confirmed', {}), g.get('pico_mml_rejected', set())),
            'UOU_LGR':            _lags(g.get('lgr_confirmed',      {}), g.get('lgr_rejected',      set())),
        },
        'rejected': {
            'LANL_aerisultra321': sorted(g.get('u321_mml_rejected', set())),
            'LANL_aerispico017':  sorted(g.get('pico_mml_rejected', set())),
            'UOU_LGR':            sorted(g.get('lgr_rejected',      set())),
        },
        'gps_corrections': g.get('gps_corrections', {}),
    }
    with open(STAGE_03_DIR / 'lag_offsets_mml.json', 'w') as fh:
        json.dump(manifest, fh, indent=2)


print('Helpers loaded.')

In [ ]:
def pre_reject_from_manifest(files, instrument, quality_manifest, cascade_bad_dates=None):
    """
    Return a set of file stems to pre-reject before the alignment widget.

    Two sources of pre-rejection:
    1. File itself is marked 'bad' in the quality manifest.
    2. Its date falls in cascade_bad_dates (Anem was bad → spike alignment impossible).
    """
    direct  = set()
    cascade = set()
    for f in files:
        status, _ = file_quality(quality_manifest, instrument, f)
        if status == 'bad':
            direct.add(f.stem)
        elif cascade_bad_dates and date_tag(f) in cascade_bad_dates:
            cascade.add(f.stem)
    if direct:
        print(f'  Pre-rejected (survey: bad) — {len(direct)} file(s):')
        for stem in sorted(direct): print(f'    {stem}')
    if cascade:
        print(f'  Pre-rejected (Anem bad on date) — {len(cascade)} file(s):')
        for stem in sorted(cascade): print(f'    {stem}')
    return direct | cascade


def make_mml_review_widget(
    gas_files, anem_dir, gps_dir, test_name,
    confirmed, rejected, save_fn=None, test_cols=None,
    secondary_refs=None, quality_tags=None, pre_rejected=None,
):
    """
    MML tube-delay review widget.

    Reference traces (gray solid):  Anem u_ms, v_ms, w_ms, RH_pct, pressure_mbar
                                     (z-scored, always shown)
    Reference trace (teal dotted):  GPS speed_ms (z-scored, always shown)
    Secondary refs  (dashed blue):  already-aligned gas series (e.g. Ultra321 H2O_ppm)
    Test traces     (colored):      gas columns — visibility controlled by checkboxes

    All traces z-scored; positive lag = gas timestamps ahead of UTC.

    RH_pct is a second breath-spike proxy (same mechanism as gas H2O_ppm — exhaled
    breath raises local humidity at the Anem inlet). pressure_mbar is included for
    context only; it is dominated by slow elevation drift, not transient spikes, so
    it is not expected to help with lag correlation.

    secondary_refs : dict[str, pd.Series] | None
        Already-aligned gas series from Stage 03 output.
    quality_tags   : dict[stem, {status,reason}] | None
        Survey tags for title annotation.
    pre_rejected   : set[str] | None
        Stems auto-skipped; Commit overrides.
    """
    if not gas_files:
        print(f'{test_name}: no files to review')
        return

    state      = {'idx': 0}
    _test_cols = list(test_cols or ['H2O_ppm', 'CH4_ppm'])
    _sec       = secondary_refs or {}
    _qual      = quality_tags or {}
    _pre_rej   = set(pre_rejected or [])

    def _build_bounds(glob_iter):
        out = []
        for f in sorted(glob_iter):
            idx = pd.read_parquet(f, columns=[]).index
            if len(idx) >= 2:
                out.append((idx[0], idx[-1], f))
        return out

    _anem_bounds = _build_bounds(anem_dir.glob('*.parquet'))
    _gps_bounds  = _build_bounds(gps_dir.glob('*.parquet'))

    _ANEM_COLS   = ['u_ms', 'v_ms', 'w_ms', 'RH_pct', 'pressure_mbar']
    _ANEM_NAMES  = ['Anem u', 'Anem v', 'Anem w', 'Anem RH', 'Anem P']
    _ANEM_COLORS = ['#888888', '#AAAAAA', '#666666', '#1F9E89', '#B8860B']
    _GPS_COLOR   = '#2E86AB'
    _SEC_COLORS  = ['#5DADE2', '#85C1E9', '#7FB3D3']
    _TEST_COLORS = ['#E67E22', '#2980B9', '#27AE60', '#8E44AD']
    _GPS_IDX = len(_ANEM_COLS)      # GPS speed trace sits right after the anem traces
    _N_REF   = _GPS_IDX + 1         # anem cols + 1 gps
    _N_SEC   = len(_sec)

    fig = go.FigureWidget(layout=go.Layout(
        autosize=True, height=420,
        margin=dict(l=55, r=10, t=10, b=30),
        yaxis=dict(title='normalized'),
        legend=dict(x=1.01, y=1, xanchor='left', font=dict(size=10)),
        hovermode='x unified',
    ))
    for i, name in enumerate(_ANEM_NAMES):
        fig.add_scatter(name=name, line=dict(color=_ANEM_COLORS[i], width=2.0))
    fig.add_scatter(name='GPS speed',
                    line=dict(color=_GPS_COLOR, width=1.5, dash='dot'), opacity=0.85)
    for i, sk in enumerate(_sec.keys()):
        fig.add_scatter(name=f'{sk} ▸aligned',
                        line=dict(color=_SEC_COLORS[i % len(_SEC_COLORS)], width=1.5, dash='dash'),
                        opacity=0.80)
    for i, col in enumerate(_test_cols):
        fig.add_scatter(name=col,
                        line=dict(color=_TEST_COLORS[i % len(_TEST_COLORS)], width=2.5))

    title_html = widgets.HTML(value='')
    lag_slider = widgets.FloatSlider(
        value=0.0, min=-MAX_LAG_S, max=MAX_LAG_S, step=0.1,
        description='Lag (s):', continuous_update=True, readout_format='.1f',
        layout=widgets.Layout(width='100%'), style={'description_width': '60px'},
    )
    btn_prev   = widgets.Button(description='◀ Prev',          layout=widgets.Layout(width='88px'))
    btn_next   = widgets.Button(description='Next ▶',          layout=widgets.Layout(width='88px'))
    btn_commit = widgets.Button(description='Commit & Next',   button_style='success',
                                layout=widgets.Layout(width='100%', height='34px'))
    btn_bad    = widgets.Button(description='Mark Bad & Next', button_style='danger',
                                layout=widgets.Layout(width='100%', height='34px'))

    # Column checkboxes — toggle which gas test traces are visible
    col_boxes = [
        widgets.Checkbox(value=True, description=col, indent=False,
                         layout=widgets.Layout(width='auto'),
                         style={'description_width': 'initial'})
        for col in _test_cols
    ]
    col_selector = widgets.VBox(col_boxes, layout=widgets.Layout(gap='2px'))

    log = widgets.Output(layout=widgets.Layout(
        max_height='80px', overflow_y='auto', border='1px solid #ddd', padding='4px',
    ))

    def _active_idxs():
        return {i for i, cb in enumerate(col_boxes) if cb.value}

    def _zscore(s):
        mu, sig = s.mean(), s.std()
        return (s - mu) / sig if sig > 0 else s * 0.0

    def _load_anem_window(t0, t1):
        parts = {c: [] for c in _ANEM_COLS}
        for fa0, fa1, f in _anem_bounds:
            if fa1 < t0 or fa0 > t1: continue
            df = pd.read_parquet(f, columns=_ANEM_COLS).sort_index()[t0:t1]
            for c in _ANEM_COLS:
                parts[c].append(df[c].dropna())
        result = {}
        for c in _ANEM_COLS:
            if parts[c]:
                s = pd.concat(parts[c]).sort_index()
                result[c] = resample_series(s[~s.index.duplicated(keep='first')])
        return result

    def _load_gps_speed_window(t0, t1):
        parts = []
        for fg0, fg1, f in _gps_bounds:
            if fg1 < t0 or fg0 > t1: continue
            df = pd.read_parquet(f, columns=['speed_ms']).sort_index()[t0:t1]
            parts.append(df['speed_ms'].dropna())
        if not parts: return None
        s = pd.concat(parts).sort_index()
        return resample_series(s[~s.index.duplicated(keep='first')])

    def _update_fig(idx, lag_s):
        active = _active_idxs()
        if idx >= len(gas_files):
            with fig.batch_update():
                for trace in fig.data:
                    trace.x = []; trace.y = []
            title_html.value = (
                f'<b>{test_name} — complete</b>  '
                f'<span style="color:#888">({len(confirmed)} committed, {len(rejected)} rejected)</span>'
            )
            return

        f           = gas_files[idx]
        key         = f.stem
        qual_status = _qual.get(key, {}).get('status', '')

        try:
            test_dict = {col: resample_series(load_parquet_col(f, col)) for col in _test_cols}
        except Exception as e:
            title_html.value = f'<span style="color:red">ERROR loading {f.name}: {e}</span>'
            return

        first_sig = next(iter(test_dict.values()))
        t0 = first_sig.index[0]  - pd.Timedelta(minutes=10)
        t1 = first_sig.index[-1] + pd.Timedelta(minutes=10)

        anem_data = _load_anem_window(t0, t1)
        gps_speed = _load_gps_speed_window(t0, t1)

        with fig.batch_update():
            fig.layout.xaxis.autorange = True
            fig.layout.yaxis.autorange = True
            for i, col in enumerate(_ANEM_COLS):
                s = anem_data.get(col)
                if s is not None and len(s) > 0:
                    z = _zscore(s)
                    fig.data[i].x = z.index.tolist(); fig.data[i].y = z.values.tolist()
                else:
                    fig.data[i].x = []; fig.data[i].y = []
            if gps_speed is not None and len(gps_speed) > 0:
                z = _zscore(gps_speed)
                fig.data[_GPS_IDX].x = z.index.tolist(); fig.data[_GPS_IDX].y = z.values.tolist()
            else:
                fig.data[_GPS_IDX].x = []; fig.data[_GPS_IDX].y = []
            for i, (sk, sseries) in enumerate(_sec.items()):
                sec_win = sseries[t0:t1]
                if not sec_win.empty:
                    z = _zscore(sec_win)
                    fig.data[_N_REF + i].x = z.index.tolist()
                    fig.data[_N_REF + i].y = z.values.tolist()
                else:
                    fig.data[_N_REF + i].x = []; fig.data[_N_REF + i].y = []
            for i, (col, sig) in enumerate(test_dict.items()):
                z       = _zscore(sig)
                shifted = (sig.index + pd.Timedelta(seconds=lag_s)).tolist()
                ti      = _N_REF + _N_SEC + i
                fig.data[ti].x       = shifted
                fig.data[ti].y       = z.values.tolist()
                fig.data[ti].name    = f'{col} ({lag_s:+.1f}s)'
                fig.data[ti].visible = i in active

        status_tag = (
            f'  <span style="color:#1E8449">✓ {confirmed[key]:+.1f}s</span>' if key in confirmed else
            f'  <span style="color:#C0392B">✗ rejected</span>'               if key in rejected  else ''
        )
        pre_tag  = (' <span style="color:#C0392B">[PRE-BAD]</span>'
                    if key in _pre_rej and key not in confirmed else '')
        qual_tag = {'uncertain': ' <span style="color:#D35400">[?]</span>',
                    'bad':       ' <span style="color:#C0392B">[survey:bad]</span>'}.get(qual_status, '')
        n_anem   = len(next(iter(anem_data.values()))) if anem_data else 0
        dtag     = date_tag(f)
        subtitle = (f'20{dtag[:2]}-{dtag[2:4]}-{dtag[4:6]}  '
                    f'{len(first_sig):,} rows  '
                    f'{first_sig.index[0].strftime("%H:%M")}–{first_sig.index[-1].strftime("%H:%M")} UTC  '
                    f'anem: {n_anem:,} pts')
        if _sec:
            subtitle += '  sec: ' + ', '.join(f'{k}: {len(v[t0:t1]):,}' for k, v in _sec.items())
        title_html.value = (
            f'<b>[{idx+1}/{len(gas_files)}]  {f.name}</b>'
            f'{pre_tag}{qual_tag}{status_tag}'
            f'<br><small style="color:#777">{subtitle}</small>'
        )

    def go_to(idx):
        if 0 <= idx < len(gas_files):
            key = gas_files[idx].stem
            if key in _pre_rej and key not in confirmed:
                with log: print(f'auto-skip  {key}  (pre-rejected)')
                state['idx'] = idx + 1; go_to(state['idx']); return
            lag_slider.value = confirmed.get(key, 0.0)
        _update_fig(idx, lag_slider.value)

    lag_slider.observe(lambda c: _update_fig(state['idx'], c['new']), names='value')
    for cb in col_boxes:
        cb.observe(lambda c: _update_fig(state['idx'], lag_slider.value), names='value')

    def on_prev(_): state['idx'] = max(0, state['idx'] - 1); go_to(state['idx'])
    def on_next(_): state['idx'] += 1; go_to(state['idx'])

    def on_commit(_):
        if state['idx'] >= len(gas_files): return
        key = gas_files[state['idx']].stem
        lag = round(lag_slider.value, 1)
        confirmed[key] = lag; rejected.discard(key); _pre_rej.discard(key)
        if save_fn: save_fn()
        with log: print(f'COMMITTED  {key}  {lag:+.1f}s')
        state['idx'] += 1; go_to(state['idx'])

    def on_bad(_):
        if state['idx'] >= len(gas_files): return
        key = gas_files[state['idx']].stem
        rejected.add(key); confirmed.pop(key, None)
        if save_fn: save_fn()
        with log: print(f'REJECTED   {key}')
        state['idx'] += 1; go_to(state['idx'])

    btn_prev.on_click(on_prev); btn_next.on_click(on_next)
    btn_commit.on_click(on_commit); btn_bad.on_click(on_bad)

    _div    = lambda: widgets.HTML('<hr style="margin:6px 0;border:none;border-top:1px solid #ddd">')
    nav_row = widgets.HBox([btn_prev, btn_next], layout=widgets.Layout(gap='6px', margin='3px 0'))
    n_pre   = len(_pre_rej)
    pre_warn = [widgets.HTML(
        f'<small style="color:#C0392B">⚠ {n_pre} pre-rejected (Commit overrides)</small>'
    )] if n_pre else []

    left_panel = widgets.VBox(
        [nav_row, _div(),
         btn_commit, btn_bad, _div(),
         widgets.HTML('<small style="color:#666">Gas columns:</small>'),
         col_selector,
         *pre_warn, log],
        layout=widgets.Layout(width='270px', min_width='270px', padding='4px 14px 4px 4px'),
    )
    right_panel = widgets.VBox(
        [title_html, fig, lag_slider],
        layout=widgets.Layout(flex='1', min_width='0', width='100%'),
    )
    display(widgets.HBox(
        [left_panel, right_panel],
        layout=widgets.Layout(width='100%', align_items='flex-start'),
    ))
    go_to(0)


print('MML review widget loaded.')


In [ ]:
quality_manifest = load_quality_manifest(QUALITY_MANIFEST_PATH)
n_tags = sum(len(v) for v in quality_manifest.values())
print(f'Quality manifest: {n_tags} entries loaded')
if not quality_manifest:
    print('  (run 03_survey.ipynb to build it — alignment will proceed without pre-rejection)')

# Dates where the Anem was marked bad — MML gas instruments on those dates
# cannot be tube-delay aligned (no spike reference) and are pre-rejected automatically.
anem_bad_dates = reference_bad_dates(quality_manifest, 'LANL_Anem', ANEM_DIR)
if anem_bad_dates:
    print(f'Anem bad on {len(anem_bad_dates)} date(s): {", ".join(sorted(anem_bad_dates))}'
          f' — dependent MML gas files will be pre-rejected')
else:
    print('Anem: no bad dates in manifest')

print(f'\nLANL_Anem:  {len(list(ANEM_DIR.glob("*.parquet")))} parquet files  (loaded lazily per gas file)')
print(f'LANL_GPS:   {len(list(GPS_DIR.glob("*.parquet")))} parquet files  (loaded lazily per gas file)')

---
## E1 — Ultra 321 MML dates vs Anem

Tube delay between the gas inlet and the Trisonica anemometer.
All traces z-scored. Anem traces (gray/teal-green/gold solid: u, v, w, RH, pressure)
and GPS speed (teal dotted) are reference. RH_pct often shows the same breath spike
as gas H2O_ppm; pressure_mbar is shown for context but rarely spikes.
Test traces (orange/blue): Ultra 321 H2O_ppm, CH4_ppm, C2H6_ppm, C3H8_ppm.

Look for a shared H2O spike and drag the lag slider until the gas traces
align with the Anem reference. No secondary gas reference yet (this is section E1).

In [ ]:
u321_all = sorted(ULTRA321_DIR.glob('*.parquet'))
u321_mml = [f for f in u321_all if ROUTING.get(raw_stem(f)) == 'MML']
print(f'Ultra 321 MML-date files: {len(u321_mml)}')
for i, f in enumerate(u321_mml):
    print(f'[{i:>2}]  {f.name}')

In [ ]:
if 'u321_mml_confirmed' not in dir(): u321_mml_confirmed = {}
if 'u321_mml_rejected'  not in dir(): u321_mml_rejected  = set()
u321_mml_pre_rejected = pre_reject_from_manifest(
    u321_mml, 'LANL_aerisultra321', quality_manifest,
    cascade_bad_dates=anem_bad_dates,
)
u321_mml_rejected |= u321_mml_pre_rejected

make_mml_review_widget(
    u321_mml, ANEM_DIR, GPS_DIR, 'Ultra321-MML',
    u321_mml_confirmed, u321_mml_rejected,
    save_fn=save_lag_offsets_mml,
    test_cols=['H2O_ppm', 'CH4_ppm', 'C2H6_ppm', 'C3H8_ppm'],
    quality_tags=quality_manifest.get('LANL_aerisultra321', {}),
    pre_rejected=u321_mml_pre_rejected,
)

---
## E2 — Pico 017 MML dates vs Anem

Same spike-alignment procedure as E1.
Secondary gas reference: Ultra 321 H2O_ppm from Stage 03 aligned output (dashed blue).
Available after the GPS correction Apply cell in Section F has run for Ultra 321.

In [ ]:
pico_all = sorted(PICO017_DIR.glob('*.parquet'))
pico_mml = [f for f in pico_all if ROUTING.get(raw_stem(f)) == 'MML']
print(f'Pico 017 MML-date files: {len(pico_mml)}')

# Load Ultra 321 aligned H2O_ppm as secondary gas reference.
# load_aligned_series reads only good files (not bad/ or bad_timestamp/) from
# STAGE_03_DIR/LANL_aerisultra321/Raw — includes both MML and WYO aligned output.
_u321_h2o_aligned = load_aligned_series(STAGE_03_DIR, 'LANL_aerisultra321', 'Raw', 'H2O_ppm')
if _u321_h2o_aligned is not None:
    _u321_h2o_aligned = resample_series(_u321_h2o_aligned)
    print(f'Secondary gas ref — Ultra321 H2O aligned: {len(_u321_h2o_aligned):,} pts')
    sec_refs_e2 = {'Ultra321 H2O aligned': _u321_h2o_aligned}
else:
    print('Secondary gas ref — Ultra321 not yet aligned (run Apply in 03b for Ultra321 first)')
    sec_refs_e2 = {}

In [ ]:
if 'pico_mml_confirmed' not in dir(): pico_mml_confirmed = {}
if 'pico_mml_rejected'  not in dir(): pico_mml_rejected  = set()
pico_mml_pre_rejected = pre_reject_from_manifest(
    pico_mml, 'LANL_aerispico017', quality_manifest,
    cascade_bad_dates=anem_bad_dates,
)
pico_mml_rejected |= pico_mml_pre_rejected

make_mml_review_widget(
    pico_mml, ANEM_DIR, GPS_DIR, 'Pico017-MML',
    pico_mml_confirmed, pico_mml_rejected,
    save_fn=save_lag_offsets_mml,
    test_cols=['H2O_ppm', 'CH4_ppm', 'C2H6_ppb'],
    secondary_refs=sec_refs_e2,
    quality_tags=quality_manifest.get('LANL_aerispico017', {}),
    pre_rejected=pico_mml_pre_rejected,
)

---
## E3 — LGR vs Anem (Mar 10 Callao Survey)

UOU LGR was present on 2026-03-10 only.  Same spike-alignment procedure.
Secondary gas reference: Ultra 321 H2O_ppm aligned (same as E2).

In [ ]:
lgr_files = sorted(LGR_DIR.glob('*.parquet'))
print(f'UOU_LGR: {len(lgr_files)} file(s)  (Mar 10 MML)')

# Reuse sec_refs_e2 (Ultra321 H2O aligned) — same reference for LGR
sec_refs_e3 = sec_refs_e2 if 'sec_refs_e2' in dir() else {}

In [ ]:
if 'lgr_confirmed' not in dir(): lgr_confirmed = {}
if 'lgr_rejected'  not in dir(): lgr_rejected  = set()
lgr_pre_rejected = pre_reject_from_manifest(
    lgr_files, 'UOU_LGR', quality_manifest,
    cascade_bad_dates=anem_bad_dates,
)
lgr_rejected |= lgr_pre_rejected

make_mml_review_widget(
    lgr_files, ANEM_DIR, GPS_DIR, 'LGR',
    lgr_confirmed, lgr_rejected,
    save_fn=save_lag_offsets_mml,
    test_cols=['H2O_ppm', 'CH4_ppm', 'CO2_ppm'],
    secondary_refs=sec_refs_e3,
    quality_tags=quality_manifest.get('UOU_LGR', {}),
    pre_rejected=lgr_pre_rejected,
)

---
## F — GPS clock correction

Reads Stage 02 `LANL_GPS` Parquet files which contain both `epoch`
(toughbook Unix timestamp) and `gps_receiver_utc` (true GPS satellite UTC).

`gps_correction[date_tag] = median(epoch − GPS_UTC_s)`

Positive value = toughbook was running fast on that date.
Files with `|offset| > MAX_GPS_CORR_S` (GPS not yet locked) are excluded.

**No widget needed** — GPS UTC is the ground truth; correction is automatic.

In [ ]:
gps_files = sorted(GPS_DIR.glob('*.parquet'))
print(f'GPS files: {len(gps_files)}\n')

raw_by_date = {}
hdr = f"{'DATE':>8}  {'FILE':<50}  {'N':>6}  {'MEDIAN_OFFSET_S':>16}  STATUS"
print(hdr)
print('-' * len(hdr))
for f in gps_files:
    df = pd.read_parquet(f, columns=['epoch', 'gps_receiver_utc'])
    df = df.dropna(subset=['gps_receiver_utc'])
    if df.empty:
        print(f'          {f.name:<50}  {"---":>6}  {"no GPS fix":>16}')
        continue
    epoch_s   = df['epoch'].astype(float)
    gps_utc_s = df['gps_receiver_utc'].apply(lambda t: t.timestamp())
    offset    = epoch_s - gps_utc_s
    med       = float(offset.median())
    dtag      = df.index[0].strftime('%y%m%d')
    ok        = abs(med) <= MAX_GPS_CORR_S
    status    = 'OK' if ok else f'SKIP (|offset|={abs(med):.0f}s > {MAX_GPS_CORR_S}s)'
    print(f'  {dtag}  {f.name:<50}  {len(offset):>6}  {med:>+16.2f}s  {status}')
    if ok:
        raw_by_date.setdefault(dtag, []).append(med)

gps_corrections = {
    tag: round(float(np.mean(vals)), 3)
    for tag, vals in raw_by_date.items()
}
print(f'\nPer-date GPS corrections (s; positive = toughbook fast):')
for tag, corr in sorted(gps_corrections.items()):
    print(f'  {tag}:  {corr:>+8.3f}s')

---
## Save lag_offsets_mml.json

Saves tube lags (E sections) and GPS corrections (F) together.
Also auto-saved after every Commit / Mark Bad click.

In [ ]:
save_lag_offsets_mml()
lag_path = STAGE_03_DIR / 'lag_offsets_mml.json'
print(f'Saved -> {lag_path}\n')
with open(lag_path) as fh:
    saved = json.load(fh)
print(f"GPS corrections: {saved['gps_corrections']}\n")
for inst, lags in saved['tube_lags'].items():
    rej = saved['rejected'].get(inst, [])
    if lags or rej:
        print(f'{inst}: {len(lags)} tube lags confirmed, {len(rej)} rejected')
        for stem, lag in sorted(lags.items()):
            print(f'  {stem:<55}  {lag:>+6.1f}s')

---
## Apply lags → `03_instrument_aligned/`

**Gas instruments** (Ultra 321, Pico 017, LGR):
  `total_lag = tube_lag − gps_corr`  → `ts_status='gps_corrected'`  `lag_ref='LANL_Anem'`

**Anem and GPS:**
  `total_lag = −gps_corr`  → `ts_status='gps_corrected'`  `lag_ref='GPS_satellite_UTC'`

Loads `lag_offsets_mml.json` — safe to re-run.

In [ ]:
def apply_gas_mml(instrument, subdirs, tube_lags, rejected_stems, gps_corrections,
                  apply_spectra=True):
    src_inst = STAGE_02_DIR / instrument
    dst_inst = STAGE_03_DIR / instrument
    n_ok = n_bad = n_warn = 0
    for subdir in subdirs:
        if not apply_spectra and subdir in ('Spectra', 'Spectralite'):
            continue
        src_dir = src_inst / subdir if subdir else src_inst
        if not src_dir.exists():
            continue
        for path in sorted(src_dir.glob('*.parquet')):
            if ROUTING.get(raw_stem(path), 'MML') != 'MML':
                continue
            rs       = raw_stem(path)
            dst_base = dst_inst / subdir if subdir else dst_inst
            if rs in rejected_stems:
                dst_path = dst_base / 'bad' / path.name
                dst_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, dst_path)
                print(f'  [BAD]  {path.name:<55}  -> bad/')
                n_bad += 1
                continue
            tube_lag = tube_lags.get(rs)
            dtag     = date_tag(path)
            gps_corr = gps_corrections.get(dtag, 0.0)
            if tube_lag is None:
                print(f'  [WARN no tube lag]  {path.name} — using 0s')
                tube_lag = 0.0; n_warn += 1
            total = tube_lag - gps_corr
            dst_path = dst_base / path.name
            rows = apply_lag_to_parquet(
                path, total, dst_path,
                ts_status='gps_corrected',
                lag_ref='LANL_Anem',
            )
            print(f'  [OK]  {path.name:<55}  '
                  f'tube={tube_lag:>+5.1f}s  gps={gps_corr:>+6.3f}s  '
                  f'total={total:>+8.3f}s  [{rows:,}]')
            n_ok += 1
    print(f'  -> {instrument}: aligned={n_ok}, bad={n_bad}, warn={n_warn}')
    return {'ok': n_ok, 'bad': n_bad, 'warn': n_warn}

print('MML apply helpers loaded.')

In [ ]:
APPLY_SPECTRA = True

with open(STAGE_03_DIR / 'lag_offsets_mml.json') as fh:
    saved = json.load(fh)

tube_lags_by_inst = saved['tube_lags']
rejected_by_inst  = {k: set(v) for k, v in saved['rejected'].items()}
gps_corrections   = saved['gps_corrections']

apply_stats = {}

# ── Gas instruments: tube delay + GPS correction ──────────────────────────────
GAS_INSTRUMENTS = {
    'LANL_aerisultra321': ['Raw', 'Eng', 'Spectra'],
    'LANL_aerispico017':  ['Raw', 'Eng', 'Spectra'],
    'UOU_LGR':            [''],
}
for inst, subdirs in GAS_INSTRUMENTS.items():
    print(f'\n{"="*60}\n  {inst}\n{"="*60}')
    stats = apply_gas_mml(
        inst, subdirs,
        tube_lags_by_inst.get(inst, {}),
        rejected_by_inst.get(inst, set()),
        gps_corrections,
        apply_spectra=APPLY_SPECTRA,
    )
    apply_stats[inst] = stats

# ── LANL_Anem: GPS correction only ───────────────────────────────────────────
print(f'\n{"="*60}\n  LANL_Anem (GPS correction only)\n{"="*60}')
n_anem = 0
for f in sorted(ANEM_DIR.glob('*.parquet')):
    dtag  = date_tag(f)
    corr  = gps_corrections.get(dtag, 0.0)
    total = -corr
    rows  = apply_lag_to_parquet(
        f, total, STAGE_03_DIR / 'LANL_Anem' / f.name,
        ts_status='gps_corrected',
        lag_ref='GPS_satellite_UTC',
    )
    print(f'  [OK]  {f.name:<55}  gps={corr:>+6.3f}s  total={total:>+8.3f}s  [{rows:,}]')
    n_anem += 1
apply_stats['LANL_Anem'] = {'ok': n_anem}

# ── LANL_GPS: GPS correction only ────────────────────────────────────────────
print(f'\n{"="*60}\n  LANL_GPS (GPS correction only)\n{"="*60}')
n_gps = 0
for f in sorted(GPS_DIR.glob('*.parquet')):
    dtag  = date_tag(f)
    corr  = gps_corrections.get(dtag, 0.0)
    total = -corr
    rows  = apply_lag_to_parquet(
        f, total, STAGE_03_DIR / 'LANL_GPS' / f.name,
        ts_status='gps_corrected',
        lag_ref='GPS_satellite_UTC',
    )
    print(f'  [OK]  {f.name:<55}  gps={corr:>+6.3f}s  total={total:>+8.3f}s  [{rows:,}]')
    n_gps += 1
apply_stats['LANL_GPS'] = {'ok': n_gps}

check_clean(REPO_ROOT, context='03b_apply_mml')
regen_git_hash, regen_git_dirty = git_info(REPO_ROOT)
apply_manifest = {
    'stage':           '03b_apply_mml',
    'run_utc':         datetime.now(timezone.utc).isoformat(),
    'git_hash':        saved['git_hash'],    # when the alignment was DECIDED
    'git_dirty':       saved['git_dirty'],
    'regen_git_hash':  regen_git_hash,      # when this output was last (re)built
    'regen_git_dirty': regen_git_dirty,
    'apply_spectra': APPLY_SPECTRA,
    'instruments':   apply_stats,
}
apply_path = STAGE_03_DIR / 'apply_manifest_mml.json'
with open(apply_path, 'w') as fh:
    json.dump(apply_manifest, fh, indent=2)
print(f'\nMML apply complete -> {apply_path}')
print('Run no_coverage pass-through cell below to finish Stage 03b.')

---
## Pass-through: no_coverage → bad_timestamp

Stage 02 `no_coverage/` files have Mountain Time clocks.
Copied to `bad_timestamp/` unchanged.

In [ ]:
NO_COVERAGE_SUBDIRS = {
    'LANL_aerisultra321': ['Raw', 'Eng', 'Spectra'],
    'LANL_aerispico017':  ['Raw', 'Eng', 'Spectra'],
}
passthrough_stats = {}
for inst, subdirs in NO_COVERAGE_SUBDIRS.items():
    n_ok = 0
    for subdir in subdirs:
        if not APPLY_SPECTRA and subdir == 'Spectra':
            continue
        src_dir = STAGE_02_DIR / inst / subdir / 'no_coverage'
        dst_dir = STAGE_03_DIR / inst / subdir / 'bad_timestamp'
        if not src_dir.exists():
            continue
        files = sorted(src_dir.glob('*.parquet'))
        if not files:
            continue
        dst_dir.mkdir(parents=True, exist_ok=True)
        for path in files:
            shutil.copy2(path, dst_dir / path.name)
            print(f'  [PASS]  {inst}/{subdir}/bad_timestamp/{path.name}')
            n_ok += 1
    passthrough_stats[inst] = {'copied': n_ok}
    print(f'  -> {inst}: {n_ok} files -> bad_timestamp/')

apply_path = STAGE_03_DIR / 'apply_manifest_mml.json'
with open(apply_path) as fh:
    m = json.load(fh)
m['passthrough'] = passthrough_stats
with open(apply_path, 'w') as fh:
    json.dump(m, fh, indent=2)
print(f'\nno_coverage pass-through complete.')
print(f'Stage 03b complete -> {STAGE_03_DIR}')